### 📈 How a Quant Engineers a Portfolio

##### ▶️ Related Quant Guild Videos:

- [The 5 Papers That Built Modern Quant Finance](https://youtu.be/ZwS1gMGegrM)

- [I Bet You've Never Found Alpha (and I Can Prove It)](https://youtu.be/UzTJHs3-eT0)

- [Quant Ranks Retail Trading Mistakes that Blow Up Your Account](https://youtu.be/1mpNxBaBeOw)

- [Non-Stationarity and Why Market Timing Fails](https://youtu.be/7nvjrgqKjJE)

- [Quant Busts 3 Trading Myths with Math](https://youtu.be/wJfIk3VnubE)

- [How to Read Options Chains](https://youtu.be/RrRbz6oXwxE)

###### ______________________________________________________________________________________________________________________________________

##### [🚀 Master your Quantitative Skills with Quant Guild](https://quantguild.com)

##### [📚 Visit the Quant Guild Library for more Jupyter Notebooks](https://github.com/romanmichaelpaolucci/Quant-Guild-Library)

##### [📈 Interactive Brokers for Algorithmic Trading](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

##### [👾 Join the Quant Guild Discord Server](discord.com/invite/MJ4FU2c6c3)

---

##### The "Closet Beta" Asset Manager

So you majored in Finance, maybe spent some time on an asset management team and realize it *can be* just handshakes and golf swings

CAPM says you'll be compensated for bearing this market risk in the cross-section, to what degree certainly varies over time

 $$
 \mathbb{E}[r_i] = r_f + \beta_i\, (\mathbb{E}[r_m] - r_f)
 $$

This isn't about *prediction* it's about explaining why you *have generated a return in the past* and why you might in the future as your *main engine for returns*

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================================
# LOAD DATA
# ==========================================================

df = pd.read_csv("portfolio_prices.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

dates = df["Date"].reset_index(drop=True)
prices = df["SPY"].astype(float).reset_index(drop=True)

# ==========================================================
# SETTINGS
# ==========================================================

ANNUAL_FACTOR = 252

# Smooth + fast:
# - Use fewer animation frames than raw rows
# - Keep the full price path preloaded
# - Animate visibility via x/y extension only
N_FRAMES = 180
ANIMATION_SPEED = 12

BG = "#282C34"
PANEL_BG = "#363B44"
GRID = "rgba(255,255,255,0.12)"
TEXT = "#E8E8E8"
MUTED = "#B8B8B8"
SPY_COLOR = "#00D1FF"

# ==========================================================
# AXIS SCALING
# ==========================================================

if len(dates) > 1:
    x_min = dates.iloc[0]
    x_max = dates.iloc[-1]
else:
    x_min = dates.iloc[0] - pd.Timedelta(days=30)
    x_max = dates.iloc[0] + pd.Timedelta(days=30)

if prices.max() == prices.min():
    y_min = prices.iloc[0] * 0.90
    y_max = prices.iloc[0] * 1.10
else:
    y_padding = (prices.max() - prices.min()) * 0.08
    y_min = prices.min() - y_padding
    y_max = prices.max() + y_padding

# ==========================================================
# TERMINAL STATS ONLY
# ==========================================================

def compute_terminal_stats(price_series):
    price_series = pd.Series(price_series).astype(float)
    rets = price_series.pct_change().dropna()

    if len(rets) < 3:
        return {
            "sharpe": 0.0,
            "sortino": 0.0,
            "beta": 1.00,
            "mdd": 0.0,
            "return": 0.0,
            "std": 0.0,
        }

    total_return = price_series.iloc[-1] / price_series.iloc[0] - 1

    ann_std = rets.std() * np.sqrt(ANNUAL_FACTOR)

    sharpe = (
        rets.mean() / rets.std() * np.sqrt(ANNUAL_FACTOR)
        if rets.std() > 0
        else 0.0
    )

    downside = rets[rets < 0]

    sortino = (
        rets.mean() / downside.std() * np.sqrt(ANNUAL_FACTOR)
        if len(downside) > 1 and downside.std() > 0
        else 0.0
    )

    cumulative = (1 + rets).cumprod()
    running_max = cumulative.cummax()
    drawdown = cumulative / running_max - 1
    mdd = drawdown.min()

    return {
        "sharpe": sharpe,
        "sortino": sortino,
        "beta": 1.00,
        "mdd": mdd,
        "return": total_return,
        "std": ann_std,
    }


def stats_table_values(stats):
    return [
        [
            "Sharpe Ratio",
            "Sortino Ratio",
            "Beta vs SPY",
            "Max Drawdown",
            "Total Return",
            "Ann. Std Dev",
        ],
        [
            f"{stats['sharpe']:.2f}",
            f"{stats['sortino']:.2f}",
            f"{stats['beta']:.2f}",
            f"{stats['mdd']:.2%}",
            f"{stats['return']:.2%}",
            f"{stats['std']:.2%}",
        ],
    ]


def make_stats_table(stats):
    return go.Table(
        columnwidth=[1.35, 1.0],
        header=dict(
            values=["Metric", "Value"],
            fill_color=PANEL_BG,
            line_color="rgba(255,255,255,0.18)",
            font=dict(color=TEXT, size=16),
            align=["left", "right"],
            height=38,
        ),
        cells=dict(
            values=stats_table_values(stats),
            fill_color=[
                ["#363B44"] * 6,
                ["#363B44"] * 6,
            ],
            line_color="rgba(255,255,255,0.10)",
            font=dict(
                color=[
                    [MUTED] * 6,
                    [TEXT] * 6,
                ],
                size=16,
            ),
            align=["left", "right"],
            height=44,
        ),
    )


terminal_stats = compute_terminal_stats(prices)

# ==========================================================
# FRAME INDEXES
# ==========================================================

if len(df) <= N_FRAMES:
    frame_indices = list(range(len(df)))
else:
    frame_indices = np.linspace(0, len(df) - 1, N_FRAMES).astype(int)
    frame_indices = sorted(set(frame_indices.tolist()))

if frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

# ==========================================================
# FIGURE
# ==========================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.76, 0.24],
    horizontal_spacing=0.06,
    specs=[[{"type": "xy"}, {"type": "table"}]],
    subplot_titles=("SPY Price Path", "SPY Statistics"),
)

# Trace 0: animated SPY path
fig.add_trace(
    go.Scatter(
        x=[dates.iloc[0]],
        y=[prices.iloc[0]],
        mode="lines",
        line=dict(color=SPY_COLOR, width=3.2),
        hovertemplate="Date: %{x|%b %d, %Y}<br>SPY: $%{y:.2f}<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=1,
)

# Trace 1: endpoint marker
fig.add_trace(
    go.Scatter(
        x=[dates.iloc[0]],
        y=[prices.iloc[0]],
        mode="markers",
        marker=dict(size=9, color=SPY_COLOR),
        hovertemplate="Date: %{x|%b %d, %Y}<br>SPY: $%{y:.2f}<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=1,
)

# Trace 2: fixed terminal stats table
fig.add_trace(
    make_stats_table(terminal_stats),
    row=1,
    col=2,
)

# ==========================================================
# FRAMES
# ==========================================================

frames = []
slider_steps = []

for i in frame_indices:
    frame_name = f"frame_{i}"

    frames.append(
        go.Frame(
            name=frame_name,
            traces=[0, 1],
            data=[
                go.Scatter(
                    x=dates.iloc[: i + 1],
                    y=prices.iloc[: i + 1],
                ),
                go.Scatter(
                    x=[dates.iloc[i]],
                    y=[prices.iloc[i]],
                ),
            ],
            layout=go.Layout(
                annotations=[
                    dict(
                        x=0.985,
                        y=-0.14,
                        xref="paper",
                        yref="paper",
                        showarrow=False,
                        font=dict(color=TEXT, size=16),
                        text=f"Date: {dates.iloc[i].strftime('%Y')}",
                    )
                ]
            ),
        )
    )

    slider_steps.append(
        dict(
            args=[
                [frame_name],
                dict(
                    frame=dict(duration=0, redraw=False),
                    transition=dict(duration=0),
                    mode="immediate",
                ),
            ],
            label=dates.iloc[i].strftime("%Y"),
            method="animate",
        )
    )

fig.frames = frames

# ==========================================================
# LAYOUT
# ==========================================================

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor=BG,
    plot_bgcolor=BG,
    width=1200,
    height=700,
    margin=dict(l=70, r=45, t=90, b=115),
    showlegend=False,
    title=dict(
        text="SPY Price Path Animation",
        x=0.04,
        y=0.96,
        xanchor="left",
        font=dict(size=30, color=TEXT),
    ),
    annotations=[
        dict(
            x=0.985,
            y=-0.14,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(color=TEXT, size=16),
            text=f"",
        )
    ],
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            showactive=False,
            x=0.00,
            y=-0.16,
            xanchor="left",
            yanchor="top",
            pad=dict(r=10, t=30),
            buttons=[
                dict(
                    label="▶ Play",
                    method="animate",
                    args=[
                        None,
                        dict(
                            frame=dict(
                                duration=ANIMATION_SPEED,
                                redraw=False,
                            ),
                            transition=dict(duration=0),
                            fromcurrent=True,
                            mode="immediate",
                        ),
                    ],
                ),
                dict(
                    label="⏸ Pause",
                    method="animate",
                    args=[
                        [None],
                        dict(
                            frame=dict(duration=0, redraw=False),
                            transition=dict(duration=0),
                            mode="immediate",
                        ),
                    ],
                ),
            ],
        )
    ],
    sliders=[
        dict(
            active=0,
            x=0.19,
            y=-0.14,
            len=0.75,
            xanchor="left",
            yanchor="top",
            pad=dict(t=45, b=10),
            currentvalue=dict(visible=False),
            transition=dict(duration=0),
            steps=slider_steps,
        )
    ],
)

# ==========================================================
# AXES
# ==========================================================

fig.update_xaxes(
    row=1,
    col=1,
    range=[x_min, x_max],
    title_text="Date",
    title_font=dict(size=16, color=TEXT),
    tickfont=dict(size=12, color=TEXT),
    showgrid=True,
    gridcolor=GRID,
    linecolor=TEXT,
    zeroline=False,
)

fig.update_yaxes(
    row=1,
    col=1,
    range=[y_min, y_max],
    title_text="SPY Price ($)",
    title_font=dict(size=16, color=TEXT),
    tickfont=dict(size=12, color=TEXT),
    showgrid=True,
    gridcolor=GRID,
    linecolor=TEXT,
    zeroline=False,
)

fig.show()

###### ______________________________________________________________________________________________________________________________________

##### Anyone can access the SPY ETF, what are you being paid for?

Well not all managers are created equally, and depending on your goals different managers can fulfill different roles

Notice how when there is a crisis you want and or need money, you lose your job, you want to double down on the fire sale, so on and so forth

In fact, because returns compound geometrically, it takes *even more* positive returns to come back from a drawdown

 $$
 \text{Volatility Drag:} \qquad \mathbb{E}[\log(1+R)] \approx \mu - \frac{1}{2}\sigma^2
 $$

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================================
# LOAD / CREATE DATES
# ==========================================================

try:
    df = pd.read_csv("portfolio_prices.csv", parse_dates=["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    dates = df["Date"].reset_index(drop=True)
except FileNotFoundError:
    dates = pd.Series(pd.bdate_range("2021-07-01", periods=252 * 5))

# ==========================================================
# SETTINGS
# ==========================================================

ANNUAL_FACTOR = 252
START_VALUE = 100.0

MEAN_DAILY_RETURN = 0.00040
DAILY_VOL_SHOCK = 0.01200

N_FRAMES = 180
ANIMATION_SPEED = 18

BG = "#282C34"
PANEL_BG = "#363B44"
GRID = "rgba(255,255,255,0.12)"
TEXT = "#E8E8E8"
MUTED = "#B8B8B8"

SMOOTH_COLOR = "#00D1FF"
VOL_COLOR = "#FFB000"
DRAG_COLOR = "rgba(255,255,255,0.45)"

SMOOTH_WIDTH = 2.8
VOL_WIDTH = 2.8
DRAG_WIDTH = 1.6
MARKER_SIZE = 7

# ==========================================================
# BUILD VOLATILITY DRAG SERIES
# ==========================================================

n = len(dates)

if n < 3:
    raise ValueError("Need at least 3 dates to animate volatility drag.")

smooth_rets = np.full(n - 1, MEAN_DAILY_RETURN)

signs = np.where(np.arange(n - 1) % 2 == 0, 1, -1)
volatile_rets = MEAN_DAILY_RETURN + signs * DAILY_VOL_SHOCK

# Keeps arithmetic average exactly equal to the smooth path
volatile_rets = volatile_rets - volatile_rets.mean() + MEAN_DAILY_RETURN

smooth_values = START_VALUE * np.r_[1, np.cumprod(1 + smooth_rets)]
volatile_values = START_VALUE * np.r_[1, np.cumprod(1 + volatile_rets)]

smooth_values = pd.Series(smooth_values, index=dates)
volatile_values = pd.Series(volatile_values, index=dates)

# ==========================================================
# STATS
# ==========================================================

def annualized_arithmetic_return(rets):
    if len(rets) == 0:
        return 0.0
    return np.mean(rets) * ANNUAL_FACTOR


def annualized_geometric_return(values, periods):
    if periods <= 0 or values.iloc[0] <= 0 or values.iloc[-1] <= 0:
        return 0.0

    return (values.iloc[-1] / values.iloc[0]) ** (ANNUAL_FACTOR / periods) - 1


def max_drawdown(values):
    values = pd.Series(values).astype(float)
    running_max = values.cummax()
    drawdown = values / running_max - 1
    return drawdown.min()


def stats_for(values):
    values = pd.Series(values).astype(float)
    rets = values.pct_change().dropna()

    if len(rets) == 0:
        return {
            "final_value": values.iloc[-1],
            "total_return": 0.0,
            "arith_ann": 0.0,
            "geom_ann": 0.0,
            "vol": 0.0,
            "drag": 0.0,
            "mdd": 0.0,
        }

    arith = annualized_arithmetic_return(rets)
    geom = annualized_geometric_return(values, len(rets))
    vol = rets.std(ddof=1) * np.sqrt(ANNUAL_FACTOR) if len(rets) > 1 else 0.0

    return {
        "final_value": values.iloc[-1],
        "total_return": values.iloc[-1] / values.iloc[0] - 1,
        "arith_ann": arith,
        "geom_ann": geom,
        "vol": vol,
        "drag": arith - geom,
        "mdd": max_drawdown(values),
    }


def table_values(smooth_stats, volatile_stats):
    return [
        [
            "Final Value",
            "Total Return",
            "Arithmetic Ann. Return",
            "Geometric Ann. Return",
            "Annualized Volatility",
            "Volatility Drag",
            "Max Drawdown",
        ],
        [
            f"${smooth_stats['final_value']:,.2f}",
            f"{smooth_stats['total_return']:.2%}",
            f"{smooth_stats['arith_ann']:.2%}",
            f"{smooth_stats['geom_ann']:.2%}",
            f"{smooth_stats['vol']:.2%}",
            f"{smooth_stats['drag']:.2%}",
            f"{smooth_stats['mdd']:.2%}",
        ],
        [
            f"${volatile_stats['final_value']:,.2f}",
            f"{volatile_stats['total_return']:.2%}",
            f"{volatile_stats['arith_ann']:.2%}",
            f"{volatile_stats['geom_ann']:.2%}",
            f"{volatile_stats['vol']:.2%}",
            f"{volatile_stats['drag']:.2%}",
            f"{volatile_stats['mdd']:.2%}",
        ],
    ]


def make_stats_table(smooth_stats, volatile_stats):
    return go.Table(
        columnwidth=[1.65, 1.0, 1.0],
        header=dict(
            values=["Metric", "Smooth", "Volatile"],
            fill_color=PANEL_BG,
            line_color="rgba(255,255,255,0.18)",
            font=dict(color=TEXT, size=14),
            align=["left", "right", "right"],
            height=38,
        ),
        cells=dict(
            values=table_values(smooth_stats, volatile_stats),
            fill_color=[
                ["#363B44"] * 7,
                ["#363B44"] * 7,
                ["#363B44"] * 7,
            ],
            line_color="rgba(255,255,255,0.10)",
            font=dict(
                color=[
                    [MUTED] * 7,
                    [SMOOTH_COLOR] * 7,
                    [VOL_COLOR] * 7,
                ],
                size=13,
            ),
            align=["left", "right", "right"],
            height=42,
        ),
    )


# ==========================================================
# FRAME INDEXES
# ==========================================================

if len(dates) <= N_FRAMES:
    frame_indices = list(range(len(dates)))
else:
    frame_indices = np.linspace(0, len(dates) - 1, N_FRAMES).astype(int)
    frame_indices = sorted(set(frame_indices.tolist()))

if frame_indices[-1] != len(dates) - 1:
    frame_indices.append(len(dates) - 1)

# ==========================================================
# AXIS SCALING
# ==========================================================

x_min = dates.iloc[0]
x_max = dates.iloc[-1]

all_values = pd.concat([smooth_values, volatile_values])
y_padding = (all_values.max() - all_values.min()) * 0.12
y_min = all_values.min() - y_padding
y_max = all_values.max() + y_padding

# ==========================================================
# EXPLAINER ANNOTATION
# ==========================================================

def drag_annotation(i=0):
    current_drag = smooth_values.iloc[i] - volatile_values.iloc[i]

    current_drag_pct = (
        current_drag / smooth_values.iloc[i]
        if smooth_values.iloc[i] != 0
        else 0.0
    )

    return dict(
        x=0.075,
        y=0.865,
        xref="paper",
        yref="paper",
        showarrow=False,
        align="left",
        bgcolor="rgba(40,44,52,0.84)",
        bordercolor="rgba(255,255,255,0.18)",
        borderwidth=1,
        borderpad=8,
        font=dict(color=TEXT, size=14),
        text=(
            "<b>Same average return.</b><br>"
            "Choppier path compounds lower.<br>"
            f"<span style='color:{VOL_COLOR}'>Current gap:</span> "
            f"${current_drag:,.2f} ({current_drag_pct:.2%})"
        ),
    )


def date_annotation(i=0):
    return dict(
        x=0.985,
        y=-0.14,
        xref="paper",
        yref="paper",
        showarrow=False,
        font=dict(color=TEXT, size=15),
        text=f"Date: {dates.iloc[i].strftime('%Y')}",
    )


# ==========================================================
# FIGURE
# ==========================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.72, 0.28],
    horizontal_spacing=0.07,
    specs=[[{"type": "xy"}, {"type": "table"}]],
)

# Trace 0: Smooth path
fig.add_trace(
    go.Scatter(
        x=[dates.iloc[0]],
        y=[smooth_values.iloc[0]],
        mode="lines",
        line=dict(color=SMOOTH_COLOR, width=SMOOTH_WIDTH),
        name="Smooth return path",
        hovertemplate="Date: %{x|%b %d, %Y}<br>Smooth: $%{y:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

# Trace 1: Smooth endpoint
fig.add_trace(
    go.Scatter(
        x=[dates.iloc[0]],
        y=[smooth_values.iloc[0]],
        mode="markers",
        marker=dict(size=MARKER_SIZE, color=SMOOTH_COLOR),
        showlegend=False,
        hovertemplate="Date: %{x|%b %d, %Y}<br>Smooth: $%{y:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

# Trace 2: Volatile path
fig.add_trace(
    go.Scatter(
        x=[dates.iloc[0]],
        y=[volatile_values.iloc[0]],
        mode="lines",
        line=dict(color=VOL_COLOR, width=VOL_WIDTH),
        name="Volatile return path",
        hovertemplate="Date: %{x|%b %d, %Y}<br>Volatile: $%{y:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

# Trace 3: Volatile endpoint
fig.add_trace(
    go.Scatter(
        x=[dates.iloc[0]],
        y=[volatile_values.iloc[0]],
        mode="markers",
        marker=dict(size=MARKER_SIZE, color=VOL_COLOR),
        showlegend=False,
        hovertemplate="Date: %{x|%b %d, %Y}<br>Volatile: $%{y:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

# Trace 4: Drag gap line
fig.add_trace(
    go.Scatter(
        x=[dates.iloc[0], dates.iloc[0]],
        y=[volatile_values.iloc[0], smooth_values.iloc[0]],
        mode="lines",
        line=dict(color=DRAG_COLOR, width=DRAG_WIDTH, dash="dot"),
        name="Volatility drag gap",
        hovertemplate="Volatility drag gap<extra></extra>",
    ),
    row=1,
    col=1,
)

# Trace 5: Stats table
fig.add_trace(
    make_stats_table(
        stats_for(smooth_values.iloc[:1]),
        stats_for(volatile_values.iloc[:1]),
    ),
    row=1,
    col=2,
)

# ==========================================================
# FRAMES
# ==========================================================

frames = []
slider_steps = []

for i in frame_indices:
    frame_name = f"frame_{i}"

    smooth_slice = smooth_values.iloc[: i + 1]
    volatile_slice = volatile_values.iloc[: i + 1]

    smooth_stats = stats_for(smooth_slice)
    volatile_stats = stats_for(volatile_slice)

    frames.append(
        go.Frame(
            name=frame_name,
            traces=[0, 1, 2, 3, 4, 5],
            data=[
                go.Scatter(
                    x=dates.iloc[: i + 1],
                    y=smooth_values.iloc[: i + 1],
                ),
                go.Scatter(
                    x=[dates.iloc[i]],
                    y=[smooth_values.iloc[i]],
                ),
                go.Scatter(
                    x=dates.iloc[: i + 1],
                    y=volatile_values.iloc[: i + 1],
                ),
                go.Scatter(
                    x=[dates.iloc[i]],
                    y=[volatile_values.iloc[i]],
                ),
                go.Scatter(
                    x=[dates.iloc[i], dates.iloc[i]],
                    y=[volatile_values.iloc[i], smooth_values.iloc[i]],
                ),
                make_stats_table(smooth_stats, volatile_stats),
            ],
            layout=go.Layout(
                annotations=[
                    drag_annotation(i),
                    date_annotation(i),
                ]
            ),
        )
    )

    slider_steps.append(
        dict(
            args=[
                [frame_name],
                dict(
                    frame=dict(duration=0, redraw=True),
                    transition=dict(duration=0),
                    mode="immediate",
                ),
            ],
            label=dates.iloc[i].strftime("%Y"),
            method="animate",
        )
    )

fig.frames = frames

# ==========================================================
# LAYOUT
# ==========================================================

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor=BG,
    plot_bgcolor=BG,
    width=1250,
    height=720,
    margin=dict(l=70, r=45, t=105, b=120),
    showlegend=True,
    legend=dict(
        orientation="h",
        x=0.03,
        y=1.04,
        xanchor="left",
        yanchor="bottom",
        bgcolor="rgba(0,0,0,0)",
        font=dict(color=TEXT, size=13),
        itemwidth=30,
    ),
    title=dict(
        text="Volatility Drag: Same Average Return, Lower Compound Return",
        x=0.04,
        y=0.97,
        xanchor="left",
        font=dict(size=27, color=TEXT),
    ),
    annotations=[
        drag_annotation(0),
        date_annotation(0),
    ],
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            showactive=False,
            x=0.00,
            y=-0.16,
            xanchor="left",
            yanchor="top",
            pad=dict(r=10, t=30),
            buttons=[
                dict(
                    label="▶ Play",
                    method="animate",
                    args=[
                        None,
                        dict(
                            frame=dict(duration=ANIMATION_SPEED, redraw=True),
                            transition=dict(duration=0),
                            fromcurrent=True,
                            mode="immediate",
                        ),
                    ],
                ),
                dict(
                    label="⏸ Pause",
                    method="animate",
                    args=[
                        [None],
                        dict(
                            frame=dict(duration=0, redraw=True),
                            transition=dict(duration=0),
                            mode="immediate",
                        ),
                    ],
                ),
            ],
        )
    ],
    sliders=[
        dict(
            active=0,
            x=0.19,
            y=-0.14,
            len=0.75,
            xanchor="left",
            yanchor="top",
            pad=dict(t=45, b=10),
            currentvalue=dict(visible=False),
            transition=dict(duration=0),
            steps=slider_steps,
        )
    ],
)

# ==========================================================
# AXES
# ==========================================================

fig.update_xaxes(
    row=1,
    col=1,
    range=[x_min, x_max],
    title_text="Date",
    title_font=dict(size=16, color=TEXT),
    tickfont=dict(size=12, color=TEXT),
    showgrid=True,
    gridcolor=GRID,
    linecolor=TEXT,
    zeroline=False,
)

fig.update_yaxes(
    row=1,
    col=1,
    range=[y_min, y_max],
    title_text="Portfolio Value ($)",
    title_font=dict(size=16, color=TEXT),
    tickfont=dict(size=12, color=TEXT),
    showgrid=True,
    gridcolor=GRID,
    linecolor=TEXT,
    zeroline=False,
)

fig.show()

###### ______________________________________________________________________________________________________________________________________

##### So how can we reduce volatility drag?

I've said it before and I'll say it again, structurally orthogonal bets with positive expected value generate wealth and can enable the use of leverage *safely*

 $$
 \text{If } \rho_{ij}=0, \quad \text{then} \quad \text{S}^2_{\pi} = \sum_{i} \text{S}^2_i
 $$

 Let's look at adding something roughly uncorrelated with our main return engine

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy.stats import linregress

# ==========================================================
# LOAD DATA
# ==========================================================

df = pd.read_csv("portfolio_prices.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

dates = df["Date"].reset_index(drop=True)
prices = df["KMLM"].astype(float).reset_index(drop=True)

# -----------------------------------------------------------------------
# Load SPY price data for beta calculation
if "SPY" in df.columns:
    spy_prices = df["SPY"].astype(float).reset_index(drop=True)
else:
    # Try to auto-load if not in CSV, otherwise beta=nan later
    try:
        spy_data = pd.read_csv("spy_prices.csv", parse_dates=["Date"])
        spy_data = spy_data.sort_values("Date").reset_index(drop=True)
        # align to main prices df (inner join on dates just in case)
        merged = pd.merge(
            pd.DataFrame({"Date": dates}),
            spy_data[["Date", "SPY"]],
            on="Date",
            how="left",
        )
        spy_prices = merged["SPY"].astype(float).reset_index(drop=True)
    except Exception:
        spy_prices = pd.Series(np.nan, index=prices.index)

# ==========================================================
# SETTINGS
# ==========================================================

ANNUAL_FACTOR = 252

N_FRAMES = 180
ANIMATION_SPEED = 12

BG = "#282C34"
PANEL_BG = "#363B44"
GRID = "rgba(255,255,255,0.12)"
TEXT = "#E8E8E8"
MUTED = "#B8B8B8"
KMLM_COLOR = "#FF3131"  # Red

# ==========================================================
# AXIS SCALING
# ==========================================================

if len(dates) > 1:
    x_min = dates.iloc[0]
    x_max = dates.iloc[-1]
else:
    x_min = dates.iloc[0] - pd.Timedelta(days=30)
    x_max = dates.iloc[0] + pd.Timedelta(days=30)

if prices.max() == prices.min():
    y_min = prices.iloc[0] * 0.90
    y_max = prices.iloc[0] * 1.10
else:
    y_padding = (prices.max() - prices.min()) * 0.08
    y_min = prices.min() - y_padding
    y_max = prices.max() + y_padding

# ==========================================================
# TERMINAL STATS ONLY
# ==========================================================

def compute_beta(price_series, benchmark_series):
    """
    Computes beta of price_series vs. benchmark_series (e.g. KMLM vs SPY), via daily returns regression.
    If benchmark is not provided or not enough overlap, returns np.nan.
    """
    price_rets = pd.Series(price_series).astype(float).pct_change().dropna()
    bench_rets = pd.Series(benchmark_series).astype(float).pct_change().dropna()
    # align series to overlapping index
    aligned = pd.concat([price_rets, bench_rets], axis=1).dropna()
    if aligned.shape[0] < 3:
        return np.nan
    y = aligned.iloc[:, 0]
    x = aligned.iloc[:, 1]
    # linregress, slope = beta
    if x.std() == 0:
        return np.nan
    slope, intercept, r_value, p_value, std_err = linregress(x, y)
    return slope

def compute_terminal_stats(price_series, benchmark_series=None):
    price_series = pd.Series(price_series).astype(float)
    rets = price_series.pct_change().dropna()

    # Compute beta only if benchmark_series is provided and valid
    beta = np.nan
    if benchmark_series is not None:
        beta = compute_beta(price_series, benchmark_series)
        if isinstance(beta, float) and (np.isnan(beta) or np.isinf(beta)):
            beta = np.nan

    if len(rets) < 3:
        return {
            "sharpe": 0.0,
            "sortino": 0.0,
            "beta": beta if not np.isnan(beta) else 0.0,
            "mdd": 0.0,
            "return": 0.0,
            "std": 0.0,
        }

    total_return = price_series.iloc[-1] / price_series.iloc[0] - 1

    ann_std = rets.std() * np.sqrt(ANNUAL_FACTOR)

    sharpe = (
        rets.mean() / rets.std() * np.sqrt(ANNUAL_FACTOR)
        if rets.std() > 0
        else 0.0
    )

    downside = rets[rets < 0]

    sortino = (
        rets.mean() / downside.std() * np.sqrt(ANNUAL_FACTOR)
        if len(downside) > 1 and downside.std() > 0
        else 0.0
    )

    cumulative = (1 + rets).cumprod()
    running_max = cumulative.cummax()
    drawdown = cumulative / running_max - 1
    mdd = drawdown.min()

    return {
        "sharpe": sharpe,
        "sortino": sortino,
        "beta": beta if not np.isnan(beta) else 0.0,
        "mdd": mdd,
        "return": total_return,
        "std": ann_std,
    }

def stats_table_values(stats):
    return [
        [
            "Sharpe Ratio",
            "Sortino Ratio",
            "Beta vs SPY",
            "Max Drawdown",
            "Total Return",
            "Ann. Std Dev",
        ],
        [
            f"{stats['sharpe']:.2f}",
            f"{stats['sortino']:.2f}",
            # Format beta: show "N/A" if nan
            f"{stats['beta']:.2f}" if stats['beta'] is not None and not np.isnan(stats['beta']) else "N/A",
            f"{stats['mdd']:.2%}",
            f"{stats['return']:.2%}",
            f"{stats['std']:.2%}",
        ],
    ]

def make_stats_table(stats):
    return go.Table(
        columnwidth=[1.35, 1.0],
        header=dict(
            values=["Metric", "Value"],
            fill_color=PANEL_BG,
            line_color="rgba(255,255,255,0.18)",
            font=dict(color=TEXT, size=16),
            align=["left", "right"],
            height=38,
        ),
        cells=dict(
            values=stats_table_values(stats),
            fill_color=[
                ["#363B44"] * 6,
                ["#363B44"] * 6,
            ],
            line_color="rgba(255,255,255,0.10)",
            font=dict(
                color=[
                    [MUTED] * 6,
                    [TEXT] * 6,
                ],
                size=16,
            ),
            align=["left", "right"],
            height=44,
        ),
    )

terminal_stats = compute_terminal_stats(prices, spy_prices)

# ==========================================================
# FRAME INDEXES
# ==========================================================

if len(df) <= N_FRAMES:
    frame_indices = list(range(len(df)))
else:
    frame_indices = np.linspace(0, len(df) - 1, N_FRAMES).astype(int)
    frame_indices = sorted(set(frame_indices.tolist()))

if frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

# ==========================================================
# FIGURE
# ==========================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.76, 0.24],
    horizontal_spacing=0.06,
    specs=[[{"type": "xy"}, {"type": "table"}]],
    subplot_titles=("KMLM Price Path", "KMLM Statistics"),
)

# Trace 0: animated KMLM path
fig.add_trace(
    go.Scatter(
        x=[dates.iloc[0]],
        y=[prices.iloc[0]],
        mode="lines",
        line=dict(color=KMLM_COLOR, width=3.2),
        hovertemplate="Date: %{x|%b %d, %Y}<br>KMLM: $%{y:.2f}<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=1,
)

# Trace 1: endpoint marker
fig.add_trace(
    go.Scatter(
        x=[dates.iloc[0]],
        y=[prices.iloc[0]],
        mode="markers",
        marker=dict(size=9, color=KMLM_COLOR),
        hovertemplate="Date: %{x|%b %d, %Y}<br>KMLM: $%{y:.2f}<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=1,
)

# Trace 2: fixed terminal stats table
fig.add_trace(
    make_stats_table(terminal_stats),
    row=1,
    col=2,
)

# ==========================================================
# FRAMES
# ==========================================================

frames = []
slider_steps = []

for i in frame_indices:
    frame_name = f"frame_{i}"

    frames.append(
        go.Frame(
            name=frame_name,
            traces=[0, 1],
            data=[
                go.Scatter(
                    x=dates.iloc[: i + 1],
                    y=prices.iloc[: i + 1],
                ),
                go.Scatter(
                    x=[dates.iloc[i]],
                    y=[prices.iloc[i]],
                ),
            ],
            layout=go.Layout(
                annotations=[
                    dict(
                        x=0.985,
                        y=-0.14,
                        xref="paper",
                        yref="paper",
                        showarrow=False,
                        font=dict(color=TEXT, size=16),
                        text=f"Date: {dates.iloc[i].strftime('%Y')}",
                    )
                ]
            ),
        )
    )

    slider_steps.append(
        dict(
            args=[
                [frame_name],
                dict(
                    frame=dict(duration=0, redraw=False),
                    transition=dict(duration=0),
                    mode="immediate",
                ),
            ],
            label=dates.iloc[i].strftime("%Y"),
            method="animate",
        )
    )

fig.frames = frames

# ==========================================================
# LAYOUT
# ==========================================================

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor=BG,
    plot_bgcolor=BG,
    width=1200,
    height=700,
    margin=dict(l=70, r=45, t=90, b=115),
    showlegend=False,
    title=dict(
        text="KMLM Price Path Animation",
        x=0.04,
        y=0.96,
        xanchor="left",
        font=dict(size=30, color=TEXT),
    ),
    annotations=[
        dict(
            x=0.985,
            y=-0.14,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(color=TEXT, size=16),
            text=f"",
        )
    ],
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            showactive=False,
            x=0.00,
            y=-0.16,
            xanchor="left",
            yanchor="top",
            pad=dict(r=10, t=30),
            buttons=[
                dict(
                    label="▶ Play",
                    method="animate",
                    args=[
                        None,
                        dict(
                            frame=dict(
                                duration=ANIMATION_SPEED,
                                redraw=False,
                            ),
                            transition=dict(duration=0),
                            fromcurrent=True,
                            mode="immediate",
                        ),
                    ],
                ),
                dict(
                    label="⏸ Pause",
                    method="animate",
                    args=[
                        [None],
                        dict(
                            frame=dict(duration=0, redraw=False),
                            transition=dict(duration=0),
                            mode="immediate",
                        ),
                    ],
                ),
            ],
        )
    ],
    sliders=[
        dict(
            active=0,
            x=0.19,
            y=-0.14,
            len=0.75,
            xanchor="left",
            yanchor="top",
            pad=dict(t=45, b=10),
            currentvalue=dict(visible=False),
            transition=dict(duration=0),
            steps=slider_steps,
        )
    ],
)

# ==========================================================
# AXES
# ==========================================================

fig.update_xaxes(
    row=1,
    col=1,
    range=[x_min, x_max],
    title_text="Date",
    title_font=dict(size=16, color=TEXT),
    tickfont=dict(size=12, color=TEXT),
    showgrid=True,
    gridcolor=GRID,
    linecolor=TEXT,
    zeroline=False,
)

fig.update_yaxes(
    row=1,
    col=1,
    range=[y_min, y_max],
    title_text="KMLM Price ($)",
    title_font=dict(size=16, color=TEXT),
    tickfont=dict(size=12, color=TEXT),
    showgrid=True,
    gridcolor=GRID,
    linecolor=TEXT,
    zeroline=False,
)

fig.show()

---

##### Let's observe what happens when we combine these ideas now with some leverage

Taking what we've learned herein, we can develop a portfolio taking advantage of these concepts to generate a lower drawdown and higher return

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================================
# LOAD DATA
# ==========================================================

df = pd.read_csv("portfolio_prices.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

required_cols = {"Date", "SPY", "KMLM"}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(f"portfolio_prices.csv is missing required columns: {sorted(missing_cols)}")

df = df.dropna(subset=["SPY", "KMLM"]).reset_index(drop=True)

dates = df["Date"].reset_index(drop=True)
spy_prices = df["SPY"].astype(float).reset_index(drop=True)
kmlm_prices = df["KMLM"].astype(float).reset_index(drop=True)

# ==========================================================
# BUILD 70% SPY / 30% KMLM PORTFOLIO
# ==========================================================

SPY_WEIGHT = 0.70
KMLM_WEIGHT = 0.30
PORTFOLIO_COLOR = "#FFA500"

asset_returns = pd.DataFrame(
    {
        "SPY": spy_prices.pct_change(),
        "KMLM": kmlm_prices.pct_change(),
    }
).fillna(0.0)

# Daily rebalanced portfolio return stream.
portfolio_returns = SPY_WEIGHT * asset_returns["SPY"] + KMLM_WEIGHT * asset_returns["KMLM"]

# Start the portfolio at the same initial value as SPY so both paths share the same visual scale.
portfolio_prices = spy_prices.iloc[0] * (1 + portfolio_returns).cumprod()
portfolio_prices = portfolio_prices.reset_index(drop=True)

# ==========================================================
# SETTINGS
# ==========================================================

ANNUAL_FACTOR = 252

# Smooth + fast:
# - Use fewer animation frames than raw rows
# - Keep the full price path preloaded
# - Animate visibility via x/y extension only
N_FRAMES = 180
ANIMATION_SPEED = 12

BG = "#282C34"
PANEL_BG = "#363B44"
GRID = "rgba(255,255,255,0.12)"
TEXT = "#E8E8E8"
MUTED = "#B8B8B8"
SPY_COLOR = "#00D1FF"

# ==========================================================
# AXIS SCALING
# ==========================================================

all_plot_values = pd.concat([spy_prices, portfolio_prices], ignore_index=True)

if len(dates) > 1:
    x_min = dates.iloc[0]
    x_max = dates.iloc[-1]
else:
    x_min = dates.iloc[0] - pd.Timedelta(days=30)
    x_max = dates.iloc[0] + pd.Timedelta(days=30)

if all_plot_values.max() == all_plot_values.min():
    y_min = all_plot_values.iloc[0] * 0.90
    y_max = all_plot_values.iloc[0] * 1.10
else:
    y_padding = (all_plot_values.max() - all_plot_values.min()) * 0.08
    y_min = all_plot_values.min() - y_padding
    y_max = all_plot_values.max() + y_padding

# ==========================================================
# TERMINAL STATS ONLY
# ==========================================================

def compute_terminal_stats(price_series, benchmark_series=None):
    price_series = pd.Series(price_series).astype(float).reset_index(drop=True)
    rets = price_series.pct_change().dropna()

    if len(rets) < 3:
        return {
            "sharpe": 0.0,
            "sortino": 0.0,
            "beta": 0.0,
            "mdd": 0.0,
            "return": 0.0,
            "std": 0.0,
        }

    total_return = price_series.iloc[-1] / price_series.iloc[0] - 1
    ann_std = rets.std() * np.sqrt(ANNUAL_FACTOR)

    sharpe = (
        rets.mean() / rets.std() * np.sqrt(ANNUAL_FACTOR)
        if rets.std() > 0
        else 0.0
    )

    downside = rets[rets < 0]
    sortino = (
        rets.mean() / downside.std() * np.sqrt(ANNUAL_FACTOR)
        if len(downside) > 1 and downside.std() > 0
        else 0.0
    )

    cumulative = (1 + rets).cumprod()
    running_max = cumulative.cummax()
    drawdown = cumulative / running_max - 1
    mdd = drawdown.min()

    beta = 1.0
    if benchmark_series is not None:
        benchmark_series = pd.Series(benchmark_series).astype(float).reset_index(drop=True)
        benchmark_rets = benchmark_series.pct_change().dropna()
        aligned = pd.concat([rets.rename("asset"), benchmark_rets.rename("benchmark")], axis=1).dropna()

        if len(aligned) > 2 and aligned["benchmark"].var() > 0:
            beta = aligned["asset"].cov(aligned["benchmark"]) / aligned["benchmark"].var()

    return {
        "sharpe": sharpe,
        "sortino": sortino,
        "beta": beta,
        "mdd": mdd,
        "return": total_return,
        "std": ann_std,
    }


def stats_table_values(spy_stats, portfolio_stats):
    return [
        [
            "Sharpe Ratio",
            "Sortino Ratio",
            "Beta vs SPY",
            "Max Drawdown",
            "Total Return",
            "Ann. Std Dev",
        ],
        [
            f"{spy_stats['sharpe']:.2f}",
            f"{spy_stats['sortino']:.2f}",
            f"{spy_stats['beta']:.2f}",
            f"{spy_stats['mdd']:.2%}",
            f"{spy_stats['return']:.2%}",
            f"{spy_stats['std']:.2%}",
        ],
        [
            f"{portfolio_stats['sharpe']:.2f}",
            f"{portfolio_stats['sortino']:.2f}",
            f"{portfolio_stats['beta']:.2f}",
            f"{portfolio_stats['mdd']:.2%}",
            f"{portfolio_stats['return']:.2%}",
            f"{portfolio_stats['std']:.2%}",
        ],
    ]


def make_stats_table(spy_stats, portfolio_stats):
    return go.Table(
        columnwidth=[1.35, 0.95, 1.15],
        header=dict(
            values=["Metric", "SPY", "70% SPY / 30% KMLM"],
            fill_color=PANEL_BG,
            line_color="rgba(255,255,255,0.18)",
            font=dict(color=TEXT, size=15),
            align=["left", "right", "right"],
            height=38,
        ),
        cells=dict(
            values=stats_table_values(spy_stats, portfolio_stats),
            fill_color=[
                ["#363B44"] * 6,
                ["#363B44"] * 6,
                ["#363B44"] * 6,
            ],
            line_color="rgba(255,255,255,0.10)",
            font=dict(
                color=[
                    [MUTED] * 6,
                    [SPY_COLOR] * 6,
                    [PORTFOLIO_COLOR] * 6,
                ],
                size=15,
            ),
            align=["left", "right", "right"],
            height=44,
        ),
    )


spy_stats = compute_terminal_stats(spy_prices, benchmark_series=spy_prices)
portfolio_stats = compute_terminal_stats(portfolio_prices, benchmark_series=spy_prices)

# ==========================================================
# FRAME INDEXES
# ==========================================================

if len(df) <= N_FRAMES:
    frame_indices = list(range(len(df)))
else:
    frame_indices = np.linspace(0, len(df) - 1, N_FRAMES).astype(int)
    frame_indices = sorted(set(frame_indices.tolist()))

if frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

# ==========================================================
# FIGURE
# ==========================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.72, 0.28],
    horizontal_spacing=0.06,
    specs=[[{"type": "xy"}, {"type": "table"}]],
    subplot_titles=("SPY vs 70/30 SPY-KMLM Portfolio", "Terminal Statistics"),
)

# Trace 0: animated SPY path
fig.add_trace(
    go.Scatter(
        x=[dates.iloc[0]],
        y=[spy_prices.iloc[0]],
        mode="lines",
        line=dict(color=SPY_COLOR, width=3.2),
        name="SPY",
        hovertemplate="Date: %{x|%b %d, %Y}<br>SPY: $%{y:.2f}<extra></extra>",
        showlegend=True,
    ),
    row=1,
    col=1,
)

# Trace 1: SPY endpoint marker
fig.add_trace(
    go.Scatter(
        x=[dates.iloc[0]],
        y=[spy_prices.iloc[0]],
        mode="markers",
        marker=dict(size=9, color=SPY_COLOR),
        name="SPY Endpoint",
        hovertemplate="Date: %{x|%b %d, %Y}<br>SPY: $%{y:.2f}<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=1,
)

# Trace 2: animated portfolio path
fig.add_trace(
    go.Scatter(
        x=[dates.iloc[0]],
        y=[portfolio_prices.iloc[0]],
        mode="lines",
        line=dict(color=PORTFOLIO_COLOR, width=3.2),
        name="70% SPY / 30% KMLM",
        hovertemplate="Date: %{x|%b %d, %Y}<br>Portfolio: $%{y:.2f}<extra></extra>",
        showlegend=True,
    ),
    row=1,
    col=1,
)

# Trace 3: portfolio endpoint marker
fig.add_trace(
    go.Scatter(
        x=[dates.iloc[0]],
        y=[portfolio_prices.iloc[0]],
        mode="markers",
        marker=dict(size=9, color=PORTFOLIO_COLOR),
        name="Portfolio Endpoint",
        hovertemplate="Date: %{x|%b %d, %Y}<br>Portfolio: $%{y:.2f}<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=1,
)

# Trace 4: fixed terminal stats table
fig.add_trace(
    make_stats_table(spy_stats, portfolio_stats),
    row=1,
    col=2,
)

# ==========================================================
# FRAMES
# ==========================================================

frames = []
slider_steps = []

for i in frame_indices:
    frame_name = f"frame_{i}"

    frames.append(
        go.Frame(
            name=frame_name,
            traces=[0, 1, 2, 3],
            data=[
                go.Scatter(
                    x=dates.iloc[: i + 1],
                    y=spy_prices.iloc[: i + 1],
                ),
                go.Scatter(
                    x=[dates.iloc[i]],
                    y=[spy_prices.iloc[i]],
                ),
                go.Scatter(
                    x=dates.iloc[: i + 1],
                    y=portfolio_prices.iloc[: i + 1],
                ),
                go.Scatter(
                    x=[dates.iloc[i]],
                    y=[portfolio_prices.iloc[i]],
                ),
            ],
            layout=go.Layout(
                annotations=[
                    dict(
                        x=0.985,
                        y=-0.14,
                        xref="paper",
                        yref="paper",
                        showarrow=False,
                        font=dict(color=TEXT, size=16),
                        text=f"Date: {dates.iloc[i].strftime('%Y')}",
                    )
                ]
            ),
        )
    )

    slider_steps.append(
        dict(
            args=[
                [frame_name],
                dict(
                    frame=dict(duration=0, redraw=False),
                    transition=dict(duration=0),
                    mode="immediate",
                ),
            ],
            label=dates.iloc[i].strftime("%Y"),
            method="animate",
        )
    )

fig.frames = frames

# ==========================================================
# LAYOUT
# ==========================================================

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor=BG,
    plot_bgcolor=BG,
    width=1250,
    height=700,
    margin=dict(l=70, r=45, t=90, b=115),
    showlegend=True,
    legend=dict(
        x=0.02,
        y=0.98,
        bgcolor="rgba(0,0,0,0)",
        font=dict(color=TEXT, size=13),
    ),
    title=dict(
        text="SPY vs 70% SPY / 30% KMLM Portfolio Animation",
        x=0.04,
        y=0.96,
        xanchor="left",
        font=dict(size=28, color=TEXT),
    ),
    annotations=[
        dict(
            x=0.985,
            y=-0.14,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(color=TEXT, size=16),
            text=f"",
        )
    ],
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            showactive=False,
            x=0.00,
            y=-0.16,
            xanchor="left",
            yanchor="top",
            pad=dict(r=10, t=30),
            buttons=[
                dict(
                    label="▶ Play",
                    method="animate",
                    args=[
                        None,
                        dict(
                            frame=dict(
                                duration=ANIMATION_SPEED,
                                redraw=False,
                            ),
                            transition=dict(duration=0),
                            fromcurrent=True,
                            mode="immediate",
                        ),
                    ],
                ),
                dict(
                    label="⏸ Pause",
                    method="animate",
                    args=[
                        [None],
                        dict(
                            frame=dict(duration=0, redraw=False),
                            transition=dict(duration=0),
                            mode="immediate",
                        ),
                    ],
                ),
            ],
        )
    ],
    sliders=[
        dict(
            active=0,
            x=0.19,
            y=-0.14,
            len=0.75,
            xanchor="left",
            yanchor="top",
            pad=dict(t=45, b=10),
            currentvalue=dict(visible=False),
            transition=dict(duration=0),
            steps=slider_steps,
        )
    ],
)

# ==========================================================
# AXES
# ==========================================================

fig.update_xaxes(
    row=1,
    col=1,
    range=[x_min, x_max],
    title_text="Date",
    title_font=dict(size=16, color=TEXT),
    tickfont=dict(size=12, color=TEXT),
    showgrid=True,
    gridcolor=GRID,
    linecolor=TEXT,
    zeroline=False,
)

fig.update_yaxes(
    row=1,
    col=1,
    range=[y_min, y_max],
    title_text="Growth of $1 SPY Starting Value",
    title_font=dict(size=16, color=TEXT),
    tickfont=dict(size=12, color=TEXT),
    showgrid=True,
    gridcolor=GRID,
    linecolor=TEXT,
    zeroline=False,
)

fig.show()


###### ______________________________________________________________________________________________________________________________________

##### Don't comprimise your main return engine

Because we have a lower max drawdown we can start to mess around with leverage *safely*

Instead of just making volatility drag **worse** we have a hedge leg that will improve our overall trajectory

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================================
# LOAD DATA
# ==========================================================

df = pd.read_csv("portfolio_prices.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

required_cols = {"Date", "SPY", "SSO", "KMLM", "MNA", "RNR"}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(f"portfolio_prices.csv is missing required columns: {sorted(missing_cols)}")

df = df.dropna(subset=["SPY", "SSO", "KMLM", "MNA", "RNR"]).reset_index(drop=True)

dates = df["Date"].reset_index(drop=True)
spy_prices = df["SPY"].astype(float).reset_index(drop=True)
sso_prices = df["SSO"].astype(float).reset_index(drop=True)
kmlm_prices = df["KMLM"].astype(float).reset_index(drop=True)
mna_prices = df["MNA"].astype(float).reset_index(drop=True)
rnr_prices = df["RNR"].astype(float).reset_index(drop=True)

# ==========================================================
# BUILD CAPITAL-EFFICIENT SSO / KMLM / MNA / RNR PORTFOLIO
# ==========================================================

SSO_WEIGHT = 0.50
KMLM_WEIGHT = 0.30
MNA_WEIGHT = 0.20
RNR_WEIGHT = 0.20
PORTFOLIO_LABEL = ""
PORTFOLIO_COLOR = "#FFA500"

asset_returns = pd.DataFrame(
    {
        "SPY": spy_prices.pct_change(),
        "SSO": sso_prices.pct_change(),
        "KMLM": kmlm_prices.pct_change(),
        "MNA": mna_prices.pct_change(),
        "RNR": rnr_prices.pct_change(),
    }
).fillna(0.0)

# Normalize weights to sum to 1.2 (as specified: 50% + 30% + 20% + 20% = 1.2)
# This means the portfolio uses 120% gross exposure.
# Daily rebalanced capital-efficient return stream.
portfolio_returns = (
    SSO_WEIGHT * asset_returns["SSO"]
    + KMLM_WEIGHT * asset_returns["KMLM"]
    + MNA_WEIGHT * asset_returns["MNA"]
    + RNR_WEIGHT * asset_returns["RNR"]
)

# Normalize both series to growth of $1 so the chart compares total return paths,
# not the raw ETF share prices.
spy_growth = (spy_prices / spy_prices.iloc[0]).reset_index(drop=True)
portfolio_growth = (1 + portfolio_returns).cumprod().reset_index(drop=True)

# ==========================================================
# SETTINGS
# ==========================================================

ANNUAL_FACTOR = 252

# Smooth + fast:
# - Use fewer animation frames than raw rows
# - Keep the full price path preloaded
# - Animate visibility via x/y extension only
N_FRAMES = 180
ANIMATION_SPEED = 12

BG = "#282C34"
PANEL_BG = "#363B44"
GRID = "rgba(255,255,255,0.12)"
TEXT = "#E8E8E8"
MUTED = "#B8B8B8"
SPY_COLOR = "#00D1FF"

# ==========================================================
# AXIS SCALING
# ==========================================================

all_plot_values = pd.concat([spy_growth, portfolio_growth], ignore_index=True)

if len(dates) > 1:
    x_min = dates.iloc[0]
    x_max = dates.iloc[-1]
else:
    x_min = dates.iloc[0] - pd.Timedelta(days=30)
    x_max = dates.iloc[0] + pd.Timedelta(days=30)

if all_plot_values.max() == all_plot_values.min():
    y_min = all_plot_values.iloc[0] * 0.90
    y_max = all_plot_values.iloc[0] * 1.10
else:
    y_padding = (all_plot_values.max() - all_plot_values.min()) * 0.08
    y_min = all_plot_values.min() - y_padding
    y_max = all_plot_values.max() + y_padding

# ==========================================================
# TERMINAL STATS ONLY
# ==========================================================

def compute_terminal_stats(price_series, benchmark_series=None):
    price_series = pd.Series(price_series).astype(float).reset_index(drop=True)
    rets = price_series.pct_change().dropna()

    if len(rets) < 3:
        return {
            "sharpe": 0.0,
            "sortino": 0.0,
            "beta": 0.0,
            "mdd": 0.0,
            "return": 0.0,
            "std": 0.0,
        }

    total_return = price_series.iloc[-1] / price_series.iloc[0] - 1
    ann_std = rets.std() * np.sqrt(ANNUAL_FACTOR)

    sharpe = (
        rets.mean() / rets.std() * np.sqrt(ANNUAL_FACTOR)
        if rets.std() > 0
        else 0.0
    )

    downside = rets[rets < 0]
    sortino = (
        rets.mean() / downside.std() * np.sqrt(ANNUAL_FACTOR)
        if len(downside) > 1 and downside.std() > 0
        else 0.0
    )

    cumulative = (1 + rets).cumprod()
    running_max = cumulative.cummax()
    drawdown = cumulative / running_max - 1
    mdd = drawdown.min()

    beta = 1.0
    if benchmark_series is not None:
        benchmark_series = pd.Series(benchmark_series).astype(float).reset_index(drop=True)
        benchmark_rets = benchmark_series.pct_change().dropna()
        aligned = pd.concat([rets.rename("asset"), benchmark_rets.rename("benchmark")], axis=1).dropna()

        if len(aligned) > 2 and aligned["benchmark"].var() > 0:
            beta = aligned["asset"].cov(aligned["benchmark"]) / aligned["benchmark"].var()

    return {
        "sharpe": sharpe,
        "sortino": sortino,
        "beta": beta,
        "mdd": mdd,
        "return": total_return,
        "std": ann_std,
    }


def stats_table_values(spy_stats, portfolio_stats):
    return [
        [
            "Sharpe Ratio",
            "Sortino Ratio",
            "Beta vs SPY",
            "Max Drawdown",
            "Total Return",
            "Ann. Std Dev",
        ],
        [
            f"{spy_stats['sharpe']:.2f}",
            f"{spy_stats['sortino']:.2f}",
            f"{spy_stats['beta']:.2f}",
            f"{spy_stats['mdd']:.2%}",
            f"{spy_stats['return']:.2%}",
            f"{spy_stats['std']:.2%}",
        ],
        [
            f"{portfolio_stats['sharpe']:.2f}",
            f"{portfolio_stats['sortino']:.2f}",
            f"{portfolio_stats['beta']:.2f}",
            f"{portfolio_stats['mdd']:.2%}",
            f"{portfolio_stats['return']:.2%}",
            f"{portfolio_stats['std']:.2%}",
        ],
    ]


def make_stats_table(spy_stats, portfolio_stats):
    return go.Table(
        columnwidth=[1.35, 0.95, 1.15],
        header=dict(
            values=["Metric", "SPY", PORTFOLIO_LABEL],
            fill_color=PANEL_BG,
            line_color="rgba(255,255,255,0.18)",
            font=dict(color=TEXT, size=15),
            align=["left", "right", "right"],
            height=38,
        ),
        cells=dict(
            values=stats_table_values(spy_stats, portfolio_stats),
            fill_color=[
                ["#363B44"] * 6,
                ["#363B44"] * 6,
                ["#363B44"] * 6,
            ],
            line_color="rgba(255,255,255,0.10)",
            font=dict(
                color=[
                    [MUTED] * 6,
                    [SPY_COLOR] * 6,
                    [PORTFOLIO_COLOR] * 6,
                ],
                size=15,
            ),
            align=["left", "right", "right"],
            height=44,
        ),
    )


spy_stats = compute_terminal_stats(spy_growth, benchmark_series=spy_growth)
portfolio_stats = compute_terminal_stats(portfolio_growth, benchmark_series=spy_growth)

# ==========================================================
# FRAME INDEXES
# ==========================================================

if len(df) <= N_FRAMES:
    frame_indices = list(range(len(df)))
else:
    frame_indices = np.linspace(0, len(df) - 1, N_FRAMES).astype(int)
    frame_indices = sorted(set(frame_indices.tolist()))

if frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

# ==========================================================
# FIGURE
# ==========================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.72, 0.28],
    horizontal_spacing=0.06,
    specs=[[{"type": "xy"}, {"type": "table"}]],
    subplot_titles=("SPY vs Capital-Efficient Multi-Asset Portfolio", "Terminal Statistics"),
)

# Trace 0: animated SPY path
fig.add_trace(
    go.Scatter(
        x=[dates.iloc[0]],
        y=[spy_growth.iloc[0]],
        mode="lines",
        line=dict(color=SPY_COLOR, width=3.2),
        name="SPY",
        hovertemplate="Date: %{x|%b %d, %Y}<br>SPY growth: %{y:.2f}x<extra></extra>",
        showlegend=True,
    ),
    row=1,
    col=1,
)

# Trace 1: SPY endpoint marker
fig.add_trace(
    go.Scatter(
        x=[dates.iloc[0]],
        y=[spy_growth.iloc[0]],
        mode="markers",
        marker=dict(size=9, color=SPY_COLOR),
        name="SPY Endpoint",
        hovertemplate="Date: %{x|%b %d, %Y}<br>SPY growth: %{y:.2f}x<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=1,
)

# Trace 2: animated portfolio path
fig.add_trace(
    go.Scatter(
        x=[dates.iloc[0]],
        y=[portfolio_growth.iloc[0]],
        mode="lines",
        line=dict(color=PORTFOLIO_COLOR, width=3.2),
        name=PORTFOLIO_LABEL,
        hovertemplate="Date: %{x|%b %d, %Y}<br>Portfolio growth: %{y:.2f}x<extra></extra>",
        showlegend=True,
    ),
    row=1,
    col=1,
)

# Trace 3: portfolio endpoint marker
fig.add_trace(
    go.Scatter(
        x=[dates.iloc[0]],
        y=[portfolio_growth.iloc[0]],
        mode="markers",
        marker=dict(size=9, color=PORTFOLIO_COLOR),
        name="Portfolio Endpoint",
        hovertemplate="Date: %{x|%b %d, %Y}<br>Portfolio growth: %{y:.2f}x<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=1,
)

# Trace 4: fixed terminal stats table
fig.add_trace(
    make_stats_table(spy_stats, portfolio_stats),
    row=1,
    col=2,
)

# ==========================================================
# FRAMES
# ==========================================================

frames = []
slider_steps = []

for i in frame_indices:
    frame_name = f"frame_{i}"

    frames.append(
        go.Frame(
            name=frame_name,
            traces=[0, 1, 2, 3],
            data=[
                go.Scatter(
                    x=dates.iloc[: i + 1],
                    y=spy_growth.iloc[: i + 1],
                ),
                go.Scatter(
                    x=[dates.iloc[i]],
                    y=[spy_growth.iloc[i]],
                ),
                go.Scatter(
                    x=dates.iloc[: i + 1],
                    y=portfolio_growth.iloc[: i + 1],
                ),
                go.Scatter(
                    x=[dates.iloc[i]],
                    y=[portfolio_growth.iloc[i]],
                ),
            ],
            layout=go.Layout(
                annotations=[
                    dict(
                        x=0.985,
                        y=-0.14,
                        xref="paper",
                        yref="paper",
                        showarrow=False,
                        font=dict(color=TEXT, size=16),
                        text=f"Date: {dates.iloc[i].strftime('%Y')}",
                    )
                ]
            ),
        )
    )

    slider_steps.append(
        dict(
            args=[
                [frame_name],
                dict(
                    frame=dict(duration=0, redraw=False),
                    transition=dict(duration=0),
                    mode="immediate",
                ),
            ],
            label=dates.iloc[i].strftime("%Y"),
            method="animate",
        )
    )

fig.frames = frames

# ==========================================================
# LAYOUT
# ==========================================================

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor=BG,
    plot_bgcolor=BG,
    width=1250,
    height=700,
    margin=dict(l=70, r=45, t=90, b=115),
    showlegend=True,
    legend=dict(
        x=0.02,
        y=0.98,
        bgcolor="rgba(0,0,0,0)",
        font=dict(color=TEXT, size=13),
    ),
    title=dict(
        text="SPY vs Capital-Efficient Portfolio",
        x=0.04,
        y=0.96,
        xanchor="left",
        font=dict(size=28, color=TEXT),
    ),
    annotations=[
        dict(
            x=0.985,
            y=-0.14,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(color=TEXT, size=16),
            text=f"",
        )
    ],
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            showactive=False,
            x=0.00,
            y=-0.16,
            xanchor="left",
            yanchor="top",
            pad=dict(r=10, t=30),
            buttons=[
                dict(
                    label="▶ Play",
                    method="animate",
                    args=[
                        None,
                        dict(
                            frame=dict(
                                duration=ANIMATION_SPEED,
                                redraw=False,
                            ),
                            transition=dict(duration=0),
                            fromcurrent=True,
                            mode="immediate",
                        ),
                    ],
                ),
                dict(
                    label="⏸ Pause",
                    method="animate",
                    args=[
                        [None],
                        dict(
                            frame=dict(duration=0, redraw=False),
                            transition=dict(duration=0),
                            mode="immediate",
                        ),
                    ],
                ),
            ],
        )
    ],
    sliders=[
        dict(
            active=0,
            x=0.19,
            y=-0.14,
            len=0.75,
            xanchor="left",
            yanchor="top",
            pad=dict(t=45, b=10),
            currentvalue=dict(visible=False),
            transition=dict(duration=0),
            steps=slider_steps,
        )
    ],
)

# ==========================================================
# AXES
# ==========================================================

fig.update_xaxes(
    row=1,
    col=1,
    range=[x_min, x_max],
    title_text="Date",
    title_font=dict(size=16, color=TEXT),
    tickfont=dict(size=12, color=TEXT),
    showgrid=True,
    gridcolor=GRID,
    linecolor=TEXT,
    zeroline=False,
)

fig.update_yaxes(
    row=1,
    col=1,
    range=[y_min, y_max],
    title_text="Growth of $1",
    title_font=dict(size=16, color=TEXT),
    tickfont=dict(size=12, color=TEXT),
    showgrid=True,
    gridcolor=GRID,
    linecolor=TEXT,
    zeroline=False,
)

fig.show()


###### ______________________________________________________________________________________________________________________________________

#### 💭 Closing Thoughts and Future Topics

 **📑 TL;DW Executive Summary**

   - This lesson guided us through the core principles of portfolio construction from a quantitative perspective, using SPY as a working example to illustrate foundational risk and return dynamics.
   - We explored how a “closet beta” manager’s results can be largely attributed to broad market exposure, referencing CAPM as a baseline for expected returns and risk.
   - The notebook demonstrated, with step-by-step code and an animated Plotly visualization, how to calculate and visualize the growth of $1 in both a market-tracking asset and an engineered portfolio, along with important performance metrics like Sharpe Ratio, Sortino Ratio, beta, max drawdown, and total return.
   - By the end, you learned how quants evaluate and compare portfolios on a statistical basis, design effective visual summaries for investment results, and why it’s crucial to understand the sources of a portfolio’s returns before claiming true alpha.

 
**Future Topics**

Technical Videos and Other Discussions

 - Fama-French / Carhart and Factor Modeling in General
 - Hawkes Processes
 - Merton Jump Diffusion Model (and Characteristic Function Pricing, Carr-Madan 1999)
 - Market-Making Models and Simulation (Stoikov-Avellaneda)
 - My First Year as a Quant
 - Why Hedge Funds are Actually Secretive
 - Non-Markovian Models (fractional Brownian motion, Volterra Process)
 - Top 3 Uses of Linear Algebra for Quant Finance
 - Girsanov's Change of Measure
 - Rough Path Theory, Applications of Path Signatures
 - Sig-Vol Model, Calibration, and Pricing
 - Trading with Alternative Data Sources
 - Pairs Trading and Statistical Arbitrage
 - Data Cleaning & Outlier Handling in Financial Time Series
 - Practical Issues in Multi-Asset Portfolio Backtesting
 - Risk Premia Harvesting: Equity, FX, Rates

[Ideas for Interactive Brokers Apps and Tutorials](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

- How Interactive Broker's API Works (EWrapper/EClient)
- How to Backtest a Trading Strategy with Interactive Brokers
- Algorithmic Volatility Trading System

---

####  $\text{Copyright © 2026 Quant Guild} \quad \quad \quad \quad \text{Author: Roman Paolucci}$